In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Load data
df = pd.read_csv(
    "/Users/shreyaparthiban/Downloads/diabetes+130-us+hospitals+for+years+1999-2008/diabetic_data.csv"
)

ids_code = pd.read_csv(
    "/Users/shreyaparthiban/Downloads/diabetes+130-us+hospitals+for+years+1999-2008/IDS_mapping.csv"
)
df['readmitted_<30']=df['readmitted'].apply(lambda x: 1 if x=='<30' else 0)


In [3]:
#Clean and Join Lookup Tables for IDs Mapping
##############################
ids_code["ID_Type"] = pd.Series([pd.NA] * len(ids_code), dtype="string")
ids_code.loc[0,  "ID_Type"] = "admission_type_id"
ids_code.loc[9,  "ID_Type"] = "discharge_disposition_id"
ids_code.loc[41, "ID_Type"] = "admission_source_id"
ids_code["ID_Type"] = ids_code["ID_Type"].ffill()

def make_lookup(df_map, id_col_name, desc_col_name="description", new_desc_name=None):
    lu = df_map[[id_col_name, desc_col_name]].copy()

    # Convert ID column to numeric; titles become NaN
    lu[id_col_name] = pd.to_numeric(lu[id_col_name], errors="coerce")
    lu = lu.dropna(subset=[id_col_name])

    # Deduplicate IDs (keep first)
    lu = lu.drop_duplicates(subset=[id_col_name])

    # Rename description column to something merge-safe
    if new_desc_name is None:
        new_desc_name = f"{id_col_name}_desc"

    lu = lu.rename(columns={desc_col_name: new_desc_name})

    return lu

# Build each lookup
admission_type_lu = make_lookup(
    ids_code.loc[ids_code["ID_Type"] == "admission_type_id"],
    id_col_name="admission_type_id",
    new_desc_name="admission_type_desc"
)

discharge_disposition_lu = make_lookup(
    ids_code.loc[ids_code["ID_Type"] == "discharge_disposition_id"]
        .rename(columns={"admission_type_id": "discharge_disposition_id"}),
    id_col_name="discharge_disposition_id",
    new_desc_name="discharge_disposition_desc"
)

admission_source_lu = make_lookup(
    ids_code.loc[ids_code["ID_Type"] == "admission_source_id"]
        .rename(columns={"admission_type_id": "admission_source_id"}),
    id_col_name="admission_source_id",
    new_desc_name="admission_source_desc"
)

for col in ["admission_type_id", "discharge_disposition_id", "admission_source_id"]:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

df = df.merge(admission_type_lu, on="admission_type_id", how="left")
df = df.merge(discharge_disposition_lu, on="discharge_disposition_id", how="left")
df = df.merge(admission_source_lu, on="admission_source_id", how="left")

In [4]:
def categorize_diag(diag):
    if pd.isnull(diag):
        return "Missing"
    
    diag = str(diag)
    
    if diag.startswith('V'):
        return "Supplementary"
    elif diag.startswith('E'):
        return "External"
    
    try:
        diag = float(diag)
    except:
        return "Other"
    
    if 1 <= diag <= 139:
        return "Infectious"
    elif 140 <= diag <= 239:
        return "Neoplasms"
    elif 240 <= diag <= 279:
        return "Endocrine"
    elif 280 <= diag <= 289:
        return "Blood"
    elif 290 <= diag <= 319:
        return "Mental"
    elif 320 <= diag <= 359:
        return "Nervous"
    elif 360 <= diag <= 389:
        return "Eye_Ear"
    elif 390 <= diag <= 459:
        return "Circulatory"
    elif 460 <= diag <= 519:
        return "Respiratory"
    elif 520 <= diag <= 579:
        return "Digestive"
    elif 580 <= diag <= 629:
        return "Genitourinary"
    elif 630 <= diag <= 679:
        return "Pregnancy"
    elif 680 <= diag <= 709:
        return "Skin"
    elif 710 <= diag <= 739:
        return "Musculoskeletal"
    elif 740 <= diag <= 759:
        return "Congenital"
    elif 760 <= diag <= 779:
        return "Perinatal"
    elif 780 <= diag <= 799:
        return "Symptoms"
    elif 800 <= diag <= 999:
        return "Injury"
    else:
        return "Other"

for col in ['diag_1','diag_2','diag_3']:
    df[col + '_cat'] = df[col].apply(categorize_diag)

In [5]:
df.replace("?",np.nan,inplace=True)
ids_code.replace ("?",np.nan,inplace=True)
summary = pd.DataFrame({
    'column': df.columns,
    'dtype': df.dtypes.astype(str),
    'null_count': df.isnull().sum(),
    'null_pct': df.isnull().mean()
})
numeric_cols = df.select_dtypes(include=np.number).columns

summary['variance'] = np.nan
summary.loc[numeric_cols, 'variance'] = df[numeric_cols].var()
categorical_cols = df.select_dtypes(include='object').columns

def get_top_values(col, n=3):
    if df[col].apply(lambda x: isinstance(x, list)).any():
        return df[col].explode().value_counts().head(n).to_dict()
    return df[col].value_counts().head(n).to_dict()

summary['top_values'] = None
summary.loc[categorical_cols, 'top_values'] = summary.loc[categorical_cols, 'column'].apply(get_top_values)
summary = summary.sort_values(by='null_pct', ascending=False)
def safe_nunique(col):
    if col.apply(lambda x: isinstance(x, list)).any():
        return col.explode().nunique()   # count unique elements inside lists
    return col.nunique()

summary['n_unique'] = df.apply(safe_nunique)
summary.sort_values(by='n_unique')

summary

,column,dtype,null_count,null_pct,variance,top_values,n_unique
weight,weight,object,98569,0.968585,NaN,"{'[75-100)': 1336, '[50-75)': 897, '[100-125)'...",9
max_glu_serum,max_glu_serum,object,96420,0.947468,NaN,"{'Norm': 2597, '>200': 1485, '>300': 1264}",3
A1Cresult,A1Cresult,object,84748,0.832773,NaN,"{'>8': 8216, 'Norm': 4990, '>7': 3812}",3
medical_specialty,medical_specialty,object,49949,0.490822,NaN,"{'InternalMedicine': 14635, 'Emergency/Trauma'...",72
payer_code,payer_code,object,40256,0.395574,NaN,"{'MC': 32439, 'HM': 6274, 'SP': 5007}",17
admission_source_desc,admission_source_desc,object,6781,0.066633,NaN,"{' Emergency Room': 57494, ' Physician Referra...",16
admission_type_desc,admission_type_desc,object,5291,0.051992,NaN,"{'Emergency': 53990, 'Elective': 18869, 'Urgen...",7
discharge_disposition_desc,discharge_disposition_desc,object,3691,0.036269,NaN,"{'Discharged to home': 60234, 'Discharged/tran...",25
race,race,object,2273,0.022336,NaN,"{'Caucasian': 76099, 'AfricanAmerican': 19210,...",5
diag_3,diag_3,object,1423,0.013983,NaN,"{'250': 11555, '401': 8289, '276': 5175}",789


In [6]:
med_cols = [col for col in df.columns 
            if df[col].dtype == 'object' 
            and df[col].isin(['No','Steady','Up','Down']).any()]
exclude = ['change','diabetesMed','readmitted']

med_cols = [col for col in med_cols if col not in exclude]

In [7]:
#drop unneeded or low cardianlity vars, keep admission typr
df=df.drop(columns=med_cols)


In [8]:
df=df.drop(columns=['weight','max_glu_serum','A1Cresult','medical_specialty','payer_code','admission_source_desc','admission_type_id','discharge_disposition_id','admission_source_id'])

In [9]:
df

,encounter_id,patient_nbr,race,gender,age,time_in_hospital,num_lab_procedures,num_procedures,num_medications,number_outpatient,...,number_diagnoses,change,diabetesMed,readmitted,readmitted_<30,admission_type_desc,discharge_disposition_desc,diag_1_cat,diag_2_cat,diag_3_cat
0,2278392,8222157,Caucasian,Female,[0-10),1,41,0,1,0,...,1,No,No,NO,0,NaN,Not Mapped,Endocrine,Other,Other
1,149190,55629189,Caucasian,Female,[10-20),3,59,0,18,0,...,9,Ch,Yes,>30,0,Emergency,Discharged to home,Endocrine,Endocrine,Endocrine
2,64410,86047875,AfricanAmerican,Female,[20-30),2,11,5,13,2,...,6,No,Yes,NO,0,Emergency,Discharged to home,Pregnancy,Endocrine,Supplementary
3,500364,82442376,Caucasian,Male,[30-40),2,44,1,16,0,...,7,Ch,Yes,NO,0,Emergency,Discharged to home,Infectious,Endocrine,Circulatory
4,16680,42519267,Caucasian,Male,[40-50),1,51,0,8,0,...,5,Ch,Yes,NO,0,Emergency,Discharged to home,Neoplasms,Neoplasms,Endocrine
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
101761,443847548,100162476,AfricanAmerican,Male,[70-80),3,51,0,16,0,...,9,Ch,Yes,>30,0,Emergency,Discharged/transferred to SNF,Endocrine,Mental,Circulatory
101762,443847782,74694222,AfricanAmerican,Female,[80-90),5,33,3,18,0,...,9,No,Yes,NO,0,Emergency,Discharged/transferred to ICF,Digestive,Endocrine,Symptoms
101763,443854148,41088789,Caucasian,Male,[70-80),1,53,0,9,1,...,13,Ch,Yes,NO,0,Emergency,Discharged to home,Infectious,Genitourinary,Mental
101764,443857166,31693671,Caucasian,Female,[80-90),10,45,2,21,0,...,9,Ch,Yes,NO,0,Urgent,Discharged/transferred to SNF,Injury,Blood,Injury


In [10]:
df.to_csv('clean_uci_diabetes.csv')